# Building Intelligent Contact Centers with Amazon Transcribe, Amazon Comprehend and Amazon Quicksight

## Setup variables

In [ ]:
import pandas as pd
import webbrowser, os
import json
import boto3
import re
import sagemaker
import time
import uuid
from sagemaker import get_execution_role
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

In [ ]:
bucket = '<your-S3-bucket>'
prefix = 'ent-innov-aws-ai/Chapter-04/'

In [ ]:
# Amazon S3 (S3) client
s3 = boto3.client('s3')
s3_resource = boto3.resource('s3')
try:
    s3.head_bucket(Bucket=bucket)
except:
    print("The S3 bucket name {} you entered seems to be incorrect, please try again".format(bucket))

In [ ]:
OUTPUT_PATH_TRANSCRIBE = 'transcribe/output'

In [ ]:
# Amazon Transcribe client
transcribe = boto3.client("transcribe")
# Amazon Comprehend client
comprehend = boto3.client("comprehend")

In [ ]:
# This is the execution role that will be used to call Amazon Transcribe
role = get_execution_role()
display(role)

## Transcribe - Run Call Analytics Job

In [ ]:
# First let us list our audio files and then upload them to the S3 bucket
audio_dir = 'call-recordings/'

for subdir, dirs, files in os.walk(audio_dir):
    for file in files:
        s3.upload_file(os.path.join(subdir, file), bucket, prefix+'transcribe/'+ os.path.join(subdir, file))
        print("Uploaded to: " + "s3://" + bucket + '/'+prefix+'transcribe/' + os.path.join(subdir, file))

#### Running call analytics job will take care of Transcription too 

In [ ]:
# Define the method that will perform transcription

def runCallAnalytics(job_name, job_uri, output_location):
    try:
        transcribe.start_call_analytics_job(
             CallAnalyticsJobName = job_name,
             Media = {
                'MediaFileUri': job_uri
             },
             DataAccessRoleArn = role,
             OutputLocation = output_location,
             ChannelDefinitions = [
                {
                    'ChannelId': 1, 
                    'ParticipantRole': 'AGENT'
                },
                {
                    'ChannelId': 0, 
                    'ParticipantRole': 'CUSTOMER'
                }
             ]
         )
        time.sleep(2)
        print("Call analytics job submitted")        
    except Exception as e:
        print(e)

In [ ]:
# Now we will loop through the recordings in our bucket to submit the transcription jobs
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket, Prefix=prefix+'transcribe/call-recordings/')
job_name_list = []
output_location = f"s3://{bucket}/{prefix}{OUTPUT_PATH_TRANSCRIBE}/"
for page in pages:
    for obj in page['Contents']:
        random = str(uuid.uuid4())
        audio_name = obj['Key'].split('/')[4].split('.')[0]
        job_name = audio_name + '-' + random
        job_name_list.append(job_name)
        job_uri = f"s3://{bucket}/{obj['Key']}"
        print('Submitting transcription for audio: ' + job_name)
        # submit the transcription job now, we will provide our current bucket name as the output bucket
        runCallAnalytics(job_name, job_uri, output_location)

#### Check job status
Please do not proceed to the next step until the job status changes to COMPLETED

In [ ]:
for job in job_name_list:
    response = transcribe.get_call_analytics_job(CallAnalyticsJobName=job_name)
    print(response['CallAnalyticsJob']['CallAnalyticsJobStatus'])

## Transcribe - Get call analytics results

You get a JSON file as an output from the call analytics job. 


In [ ]:

def upload_segments(job, i, transcript):
    # Get the turn by turn contents
    turn_idx = 0
    idx = len(turn_df)
    transcript_text = ""
    for turn in transcript['Transcript']:
        idx += 1
        turn_idx += 1
        # Build the base dataframe of call details, sentiment and loudness
        turn_df.at[idx,'job'] = job
        turn_df.at[idx, 'turn'] = turn_idx
        turn_df.at[idx,'content'] = str(turn['Content']).replace("'","").replace(",","")
        turn_df.at[idx, 'participant_role'] = turn['ParticipantRole']
        turn_df.at[idx, 'sentiment'] = turn['Sentiment']
        
        # Get an average loudness score for each turn
        tot_loud = 0
        for loud in turn['LoudnessScores']:
            if loud is not None:
                tot_loud += int(loud)
        avg_loudness = tot_loud/len(turn['LoudnessScores'])
        turn_df.at[idx, 'loudness_score'] = round(avg_loudness,0)
        
        # Now add Transcribe analytics such as issues, actions and outcomes
        if turn.get('IssuesDetected'):
            ib = turn.get('IssuesDetected')[0]['CharacterOffsets']['Begin']
            ie = turn.get('IssuesDetected')[0]['CharacterOffsets']['End']
            turn_df.at[idx, 'issue'] = str(turn['Content'][ib:ie]).replace("'","").replace(",","")
        else:
            turn_df.at[idx, 'issue'] = "No issues detected"
        if turn.get('ActionItemsDetected'):
            ib = turn.get('ActionItemsDetected')[0]['CharacterOffsets']['Begin']
            ie = turn.get('ActionItemsDetected')[0]['CharacterOffsets']['End']
            turn_df.at[idx, 'actions'] = str(turn['Content'][ib:ie]).replace("'","").replace(",","")
        else:
            turn_df.at[idx, 'actions'] = "No actions detected"
        if turn.get('OutcomesDetected'):
            ib = turn.get('OutcomesDetected')[0]['CharacterOffsets']['Begin']
            ie = turn.get('OutcomesDetected')[0]['CharacterOffsets']['End']
            turn_df.at[idx, 'outcomes'] = str(turn['Content'][ib:ie]).replace("'","").replace(",","")
        else:
            turn_df.at[idx, 'outcomes'] = "No outcomes detected"
        
       
        # Get Comprehend entities for the full text
        ent_df.at[idx,'job'] = job
        ent_df.at[idx, 'turn'] = turn_idx
        transcript_text = str(turn['Content']).replace("'","").replace(",","")
        ent_response = comprehend.detect_entities(Text=transcript_text, LanguageCode='en')
        # Initialize values
        ent_df.at[idx, 'QUANTITY'] = " "
        ent_df.at[idx, 'PERSON'] = " "
        ent_df.at[idx, 'DATE'] = " "
        ent_df.at[idx, 'ORGANIZATION'] = " "
        ent_df.at[idx, 'LOCATION'] = " "
        ent_df.at[idx, 'EVENT'] = " "
        ent_df.at[idx, 'OTHER'] = " "
        ent_df.at[idx, 'COMMERCIAL_ITEM'] = " "
        for entity in ent_response['Entities']:
            if entity.get('Type'):
                ent_df.at[idx, entity['Type']] = str(entity['Text']).replace("'","").replace(",","")
    
    # Finally get the overall call characteristics into a seperate dataframe
    call_df.at[i,'job'] = job
    call_df.at[i,'non-talk-instances'] = len(transcript['ConversationCharacteristics']['NonTalkTime']['Instances'])
    call_df.at[i,'non-talk-time'] = transcript['ConversationCharacteristics']['NonTalkTime']['TotalTimeMillis']
    call_df.at[i, 'interruption_count'] = transcript['ConversationCharacteristics']['Interruptions']['TotalCount']
    call_df.at[i, 'interruption_tot_duration'] = transcript['ConversationCharacteristics']['Interruptions']['TotalTimeMillis']
    call_df.at[i, 'total_conv_duration'] = transcript['ConversationCharacteristics']['TotalConversationDurationMillis']
    temp_talk_speed = transcript['ConversationCharacteristics']['TalkSpeed']['DetailsByParticipant']
    if temp_talk_speed.get('AGENT'):
        call_df.at[i, 'agent_talk_speed'] = temp_talk_speed.get('AGENT')['AverageWordsPerMinute']
    if temp_talk_speed.get('CUSTOMER'):
        call_df.at[i, 'customer_talk_speed'] = temp_talk_speed.get('CUSTOMER')['AverageWordsPerMinute']
    temp_talk_time = transcript['ConversationCharacteristics']['TalkTime']['DetailsByParticipant']
    if temp_talk_time.get('AGENT'):
        call_df.at[i, 'agent_talk_time'] = temp_talk_time.get('AGENT')['TotalTimeMillis']
    if temp_talk_time.get('CUSTOMER'):
        call_df.at[i, 'customer_talk_time'] = temp_talk_time.get('CUSTOMER')['TotalTimeMillis']
    

In [ ]:
# First we need an output directory
dir = os.getcwd()+'/output'
if not os.path.exists(dir):
    os.makedirs(dir)

In [ ]:
# Our transcript is URL format
import urllib3

# Get Call Analytics output
icols = ['job', 'turn','content', 'participant_role', 'loudness_score']
turn_df = pd.DataFrame(columns=icols)
ocols = ['job', 'non-talk-instances', 'non-talk-time', 'interruption_count', 'interruption_tot_duration', 'total_conv_duration', 'agent_talk_speed', 'agent_talk_time', 'customer_talk_speed', 'customer_talk_time']
call_df = pd.DataFrame(columns=ocols)
ecols = ['job','turn']
ent_df = pd.DataFrame(columns=ecols)
i = -1

for job in job_name_list:
    
    response = transcribe.get_call_analytics_job(CallAnalyticsJobName=job)
    #print(response)
    json_file = response['CallAnalyticsJob']['Transcript']['TranscriptFileUri']
    a = json_file.split('/')
    tca_prefix = '/'.join(a[4:])
    s3.download_file(bucket,tca_prefix,dir+'/'+job)
    with open(dir+'/'+job) as f:
        data = json.load(f)
    i += 1
    upload_segments(str(job), i, data)
csv_file = '.csv'
turn_df.to_csv(dir+"/turn-transcripts"+csv_file, index=False)
call_df.to_csv(dir+"/call-analytics"+csv_file, index=False)
ent_df.to_csv(dir+"/entities"+csv_file, index=False)
s3.upload_file(dir+'/turn-transcripts'+csv_file, bucket, prefix+'quicksight/data/transcripts/' + 'turn-transcripts'+csv_file)
s3.upload_file(dir+'/call-analytics'+csv_file, bucket, prefix+'quicksight/data/analytics/' + 'call-analytics'+csv_file)
s3.upload_file(dir+'/entities'+csv_file, bucket, prefix+'quicksight/data/entities/' + 'entities'+csv_file)

In [ ]:
turn_df.head()

In [ ]:
call_df.head()

In [ ]:
ent_df[35:40]

## Visualize unique insights

#### Define variables

In [ ]:
# initialize variables we need
infile = 'quicksight_raw_manifest.json'
outfile = 'quicksight_formatted_manifest_type.json'
inprefix = prefix+'quicksight/data'
manifestprefix = prefix+'quicksight/manifest'

#### List transcripts for import to QuickSight

In [ ]:
!aws s3 ls s3://{bucket}/{inprefix} --recursive 

#### Update QuickSight Manifest
You will replace the S3 bucket and prefix from the raw manifest file with the S3 prefix that contains the transcript files. A new formatted manifest file will be created for QuickSight to use

In [ ]:
# Create formatted manifests for each type of dataset we need from the raw manifest JSON
types = ['transcripts','analytics','entities']

manifest = open(infile, 'r')
ln = json.load(manifest)
t = json.dumps(ln['fileLocations'][0]['URIPrefixes'])
for type in types:
    t1 = t.replace('bucket', bucket).replace('prefix', inprefix + '/' + type)
    ln['fileLocations'][0]['URIPrefixes'] = json.loads(t1)
    outfile_rep = outfile.replace('type', type)
    with open(outfile_rep, 'w', encoding='utf-8') as out:
        json.dump(ln, out, ensure_ascii=False, indent=4)
    # Upload the manifest to S3
    s3.upload_file(outfile_rep, bucket, manifestprefix + '/' + outfile_rep)
    print("New manifest file ready at: s3://{}/{}".format(bucket, manifestprefix + '/' + outfile_rep))

## END OF NOTEBOOK, PLEASE GO BACK TO CHAPTER 4 FOR FURTHER INSTRUCTIONS